<a href="https://colab.research.google.com/github/Priya-ssharma/Capstone_Project_PriyaSharma/blob/main/Mama_Earth_Capstone_Project_Priya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from google.colab import files
up = files.upload()

Saving Data CSV.zip to Data CSV (2).zip


In [27]:
# Unzip and copy the uploaded data to the data/ folder
import zipfile
import shutil
from pathlib import Path

zip_path = Path("Data CSV.zip")
if zip_path.exists():
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")

    # Copy extracted csv files to data/
    extracted_dir = Path("Data CSV")
    if extracted_dir.exists():
        for csv_file in extracted_dir.glob("*.csv"):
            shutil.copy(csv_file, Path("data") / csv_file.name)
    else:
        # Check if they were extracted directly to the root
        for csv_file in Path(".").glob("*.csv"):
            shutil.copy(csv_file, Path("data") / csv_file.name)

    print("Files in data/ folder:", os.listdir("data"))
else:
    print("Data CSV.zip not found. Please re-upload.")

Files in data/ folder: ['customers.csv', 'products.csv', 'orders.csv']


Part 1, Task 1: SCHEMA FILE

In [28]:
%%writefile sql/schema.sql
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id VARCHAR(10) PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    city VARCHAR(50) NOT NULL,
    city_tier INT NOT NULL,
    signup_date DATE NOT NULL,
    acquisition_source VARCHAR(20) NOT NULL
);

CREATE TABLE products (
    product_id VARCHAR(10) PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category VARCHAR(30) NOT NULL,
    price DECIMAL(10,2) NOT NULL
);

CREATE TABLE orders (
    order_id VARCHAR(10) PRIMARY KEY,
    customer_id VARCHAR(10) NOT NULL,
    product_id VARCHAR(10) NOT NULL,
    order_date DATE NOT NULL,
    quantity INT NOT NULL,
    discount_pct INT,
    payment_method VARCHAR(10) NOT NULL,
    rating INT,
    returned INT NOT NULL DEFAULT 0,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

Overwriting sql/schema.sql


In [29]:
import sqlite3, pandas as pd
from pathlib import Path

if Path("mamaearth.db").exists(): Path("mamaearth.db").unlink()
con = sqlite3.connect("mamaearth.db")
def q(sql): return pd.read_sql_query(sql, con)

con.executescript(Path("sql/schema.sql").read_text())
q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,customers
1,products
2,orders


Part 1, Task 2: Seed data

In [30]:
%%writefile sql/generate_seed.py
import csv
from pathlib import Path

ROOT = Path(".")
NUMERIC = {"customers": {"city_tier"}, "products": {"price"},
           "orders": {"quantity", "discount_pct", "rating", "returned"}}

def literal(value, is_numeric):
    if value == "":
        return "NULL"
    return value if is_numeric else "'" + value.replace("'", "''") + "'"

lines = []
for table in ["customers", "products", "orders"]:
    with open(ROOT / "data" / f"{table}.csv", newline="") as f:
        for row in csv.DictReader(f):
            cols = ", ".join(row)
            vals = ", ".join(literal(v, c in NUMERIC[table]) for c, v in row.items())
            lines.append(f"INSERT INTO {table} ({cols}) VALUES ({vals});")

(ROOT / "sql" / "seed_data.sql").write_text("\n".join(lines) + "\n")
print(f"Wrote {len(lines)} INSERT statements")

Overwriting sql/generate_seed.py


In [31]:
!python sql/generate_seed.py

Wrote 241 INSERT statements


In [32]:
con.executescript(Path("sql/seed_data.sql").read_text())

In [33]:
def q(sql):
    return pd.read_sql_query(sql, con).style.hide(axis="index")

In [34]:
q("""SELECT (SELECT COUNT(*) FROM customers) AS customers,
           (SELECT COUNT(*) FROM products)  AS products,
           (SELECT COUNT(*) FROM orders)    AS orders""")

customers,products,orders
45,16,180


In [35]:
!python sql/generate_seed.py
!wc -l sql/seed_data.sql

Wrote 241 INSERT statements
241 sql/seed_data.sql


In [36]:
import sqlite3, pandas as pd
from pathlib import Path
from IPython.display import display

if Path("mamaearth.db").exists(): Path("mamaearth.db").unlink()
con = sqlite3.connect("mamaearth.db")
def q(sql): return pd.read_sql_query(sql, con).style.hide(axis="index")

con.executescript(Path("sql/schema.sql").read_text())
con.executescript(Path("sql/seed_data.sql").read_text())
con.commit()

display(q("""SELECT (SELECT COUNT(*) FROM customers) AS customers,
                    (SELECT COUNT(*) FROM products)  AS products,
                    (SELECT COUNT(*) FROM orders)    AS orders"""))

customers,products,orders
45,16,180


In [37]:
Path("sql/reports.sql").write_text("-- Mamaearth reports. Run on RAW data, fresh DB.\n\n")

def report(label, sql, note=""):
    df = pd.read_sql_query(sql, con)
    lines = ["-- " + label]
    if note: lines.append("-- " + note)
    lines.append("-- OUTPUT:")
    lines += ["--   " + l for l in df.to_string(index=False).splitlines()]
    lines.append(sql.strip().rstrip(";") + ";")
    with open("sql/reports.sql", "a") as f:
        f.write("\n".join(lines) + "\n\n")
    return df.style.hide(axis="index")

def run_change(label, sql):
    con.executescript(sql)
    with open("sql/reports.sql", "a") as f:
        f.write("-- " + label + "\n" + sql.strip() + "\n\n")


In [38]:
q("SELECT typeof(rating) AS type, COUNT(*) AS n FROM orders GROUP BY 1")

type,n
integer,165
null,15


Task3, Reports; Nine Business Queries

a.

In [39]:
report("(a) Order totals", """SELECT COUNT(*) AS total_orders,
  ROUND(SUM(o.quantity*p.price*(1-COALESCE(o.discount_pct,0)/100.0)),2) AS total_revenue,
  ROUND(AVG(o.quantity*p.price*(1-COALESCE(o.discount_pct,0)/100.0)),2) AS avg_order_value
FROM orders o JOIN products p ON o.product_id=p.product_id""")

total_orders,total_revenue,avg_order_value
180,99860.200000,554.780000


b.

In [40]:
report("(b) COUNT(*) vs COUNT(rating)", """SELECT COUNT(*) AS all_rows,
  COUNT(rating) AS rated_rows, COUNT(*)-COUNT(rating) AS unrated_rows FROM orders""")

all_rows,rated_rows,unrated_rows
180,165,15


c1

In [41]:
report("(c1) LEFT JOIN: customers with zero orders", """SELECT c.customer_id, c.name
FROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id
GROUP BY c.customer_id, c.name HAVING COUNT(o.order_id)=0""")

customer_id,name
C045,Vihaan


c2

In [42]:
report("(c2) Independent check with NOT IN", """SELECT customer_id, name FROM customers
WHERE customer_id NOT IN (SELECT DISTINCT customer_id FROM orders)""")

customer_id,name
C045,Vihaan


d.

In [43]:
report("(d) Return rate by city above 20%", """SELECT c.city, COUNT(*) AS total_orders,
  SUM(o.returned) AS returned_orders,
  ROUND(100.0*SUM(o.returned)/COUNT(*),1) AS return_rate_pct
FROM orders o JOIN customers c ON o.customer_id=c.customer_id
GROUP BY c.city HAVING return_rate_pct>20 ORDER BY return_rate_pct DESC""")

city,total_orders,returned_orders,return_rate_pct
Jaipur,19,8,42.100000
Lucknow,49,15,30.600000
Bangalore,33,8,24.200000


e,

In [44]:
base = """SELECT c.customer_id, c.name,
  ROUND(SUM(o.quantity*p.price*(1-COALESCE(o.discount_pct,0)/100.0)),2) AS total_spend
FROM orders o JOIN products p ON o.product_id=p.product_id
JOIN customers c ON o.customer_id=c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_spend DESC, c.customer_id ASC"""
note = "Tie-break on customer_id: equal spenders would otherwise come back in arbitrary order, so LIMIT/OFFSET pages could overlap or skip rows."
display(report("(e1) Top 5 by spend", base + "\nLIMIT 5", note))
display(report("(e2) Ranks 3 to 5", base + "\nLIMIT 3 OFFSET 2", note))

customer_id,name,total_spend
C043,Reyansh,12920.000000
C026,Isha,8371.600000
C008,Meera,4564.600000
C011,Arjun,4111.000000
C042,Sanya,3785.000000


customer_id,name,total_spend
C008,Meera,4564.600000
C011,Arjun,4111.000000
C042,Sanya,3785.000000


f.

In [45]:
report("(f) Revenue by category", """SELECT p.category, COUNT(*) AS order_count,
  ROUND(SUM(o.quantity*p.price*(1-COALESCE(o.discount_pct,0)/100.0)),2) AS category_revenue
FROM orders o JOIN products p ON o.product_id=p.product_id
JOIN customers c ON o.customer_id=c.customer_id
GROUP BY p.category ORDER BY category_revenue DESC""")

category,order_count,category_revenue
Haircare,54,44956.100000
Skincare,60,27346.000000
Babycare,30,16805.000000
PersonalCare,36,10753.100000


g.

In [46]:
report("(g) Names starting with A", "SELECT * FROM customers WHERE name LIKE 'A%'")

customer_id,name,city,city_tier,signup_date,acquisition_source
C001,Aarav,Mumbai,1,2026-01-07,Organic
C003,Aditi,Mumbai,1,2026-06-23,Organic
C004,Ananya,Lucknow,2,2026-01-23,Organic
C011,Arjun,Bangalore,1,2026-03-13,Referral
C021,Aryan,Bangalore,1,2026-02-11,Ad
C030,Anika,Bangalore,1,2026-02-24,Organic
C031,Aditya,Jaipur,2,2026-06-14,Ad
C036,Aisha,Delhi,1,2026-05-11,Ad
C041,Ayaan,Lucknow,2,2026-01-03,Organic
C044,Aria,Bangalore,1,2026-04-22,Referral


h.

In [47]:
report("(h) Distinct acquisition sources", "SELECT DISTINCT acquisition_source FROM customers")

acquisition_source
Organic
Referral
Ad
Social


i.

In [48]:
run_change("(i) ALTER + UPDATE with CASE", """ALTER TABLE customers ADD COLUMN loyalty_tier VARCHAR(10);
UPDATE customers SET loyalty_tier = CASE WHEN city_tier = 1 THEN 'Gold' ELSE 'Silver' END;""")
report("(i) check", "SELECT loyalty_tier, COUNT(*) AS customers FROM customers GROUP BY loyalty_tier")

loyalty_tier,customers
Gold,28
Silver,17
